# Create Kone Foundation Awards

Creates Kone Foundation (Koneen Säätiö) awards from Finland's national research-information hub **research.fi**.

**Prerequisites:** run `scripts/local/business_finland_to_s3.py --funder-name "Kone Foundation" --provenance research_fi_kone --slug kone_foundation` first (same reusable research.fi adapter; queries the public Elasticsearch `funding` index, filtered to this funder).

**Data source:** `https://researchfi-api-production.2.rahtiapp.fi/portalapi/funding/_search` (public, no auth, `search_after` pagination). **S3:** `s3a://openalex-ingest/awards/kone_foundation/kone_foundation_projects.parquet`

**OpenAlex funder:** Kone Foundation (Koneen Säätiö) · `funder_id 4320323433` · FI.

**Schema notes:**
- `funder_award_id` = `funderProjectNumber` (research.fi funding-decision id) — the real grant reference / award_number (100%).
- `amount` = EUR. 0/neg -> NULL (never fillna 0).
- `lead_investigator` = funding contact person (given/family) when published, else NULL; `affiliation.name` = recipient org (`consortiumOrganizationNameEn`), country Finland. **Foundation grants to individuals often have NULL contact-person — NULL is the correct value (do not backfill).**
- `display_name` = project name (EN preferred, else FI/SV).

**Provenance** `research_fi_kone`, **priority 215** (Claude = odd slots). Same reusable research.fi adapter as Business Finland (PR #252).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kone_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kone_foundation/kone_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.kone_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.kone_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.kone_foundation_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast
Must return exactly 1 row. If 0, STOP — funder missing from `openalex.common.funder`.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder WHERE funder_id = 4320323433;

## Step 2: Create Kone Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kone_foundation_awards
USING delta
AS
WITH
fdr AS (
    SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320323433
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'research' as funding_type,
        g.funding_type_raw as funder_scheme,
        'research_fi_kone' as provenance,
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.start_year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CASE WHEN TRY_CAST(g.end_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.end_year, '-12-31') AS DATE) ELSE NULL END as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.end_year AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.institution_name as name,
                        'Finland' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        'https://research.fi/en/results?type=funding' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.kone_foundation_raw g
    CROSS JOIN fdr f
    WHERE g.funder_award_id IS NOT NULL AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'research_fi_kone' AND priority = 215;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    215 as priority
FROM openalex.awards.kone_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.kone_foundation_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, start_year,
       lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.kone_foundation_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a freq check. Foundation contact-person is often NULL (grants to individuals) — NULL given/family is correct, not a bug. affiliation.name carries the recipient org.
SELECT lead_investigator.given_name AS g, lead_investigator.family_name AS f, COUNT(*) AS n
FROM openalex.awards.kone_foundation_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_inst,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_inst
FROM openalex.awards.kone_foundation_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as inst, COUNT(*) as n
FROM openalex.awards.kone_foundation_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20;